# Explainable AI for Medical Diagnosis — Diabetes Prediction

Companion notebook to `research/medical-diagnosis-xai.md`.

This notebook was generated with Claude Code to provide further research material for our topic.

We train a classifier to predict diabetes onset (Pima Indians Diabetes
dataset), then use **SHAP** and **LIME** to explain individual predictions,
and investigate two of the research directions from the write-up:

1. **Agreement** — do SHAP and LIME identify the same important features
   for the same predictions?
2. **Faithfulness** — if an explanation says a feature is important, does
   perturbing that feature actually move the model's prediction the way
   the explanation implies?

This is a first pass, not a finished analysis — see the `TODO`/discussion
cells throughout for what still needs interpretation or extension.

## 0. Setup

In [ ]:
# If running in Colab, uncomment:
# !pip install shap lime -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score
)

import shap
from lime.lime_tabular import LimeTabularExplainer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Dataset

Pima Indians Diabetes dataset (768 patients, 8 clinical features, binary
outcome: diabetes onset within 5 years). Loaded directly from a public
mirror so this runs in Colab with no manual download.

Features: `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`,
`Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age`. Target: `Outcome`
(1 = diabetes, 0 = no diabetes).

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
    "pima-indians-diabetes.csv"
)
COLUMNS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
    "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]

df = pd.read_csv(DATA_URL, names=COLUMNS)
df.head()

## 2. Explore Data (EDA)

In [ ]:
print(df.shape)
df.describe()

In [ ]:
# Class balance
df["Outcome"].value_counts(normalize=True).rename("proportion")

In [ ]:
# Known data quality issue with this dataset: Glucose, BloodPressure,
# SkinThickness, Insulin and BMI use 0 as a placeholder for "missing",
# which is not physiologically possible for these measurements.
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
(df[ZERO_AS_MISSING] == 0).sum().rename("num_zero_values")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df.corr(numeric_only=True)
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns)
fig.colorbar(im)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.show()

## 3. Preprocess

Replace invalid zeros with `NaN` and impute with the median (computed on
the training split only, to avoid leakage), then split into train/test.

In [ ]:
df_clean = df.copy()
df_clean[ZERO_AS_MISSING] = df_clean[ZERO_AS_MISSING].replace(0, np.nan)

X = df_clean.drop(columns="Outcome")
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

X_train.shape, X_test.shape

## 4. Train Model

A `RandomForestClassifier` as our black-box-ish model — expressive enough
that XAI is warranted, but simple enough that we can sanity-check SHAP/LIME
against its actual (tree-based) feature importances.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300, max_depth=6, random_state=RANDOM_STATE
)
model.fit(X_train, y_train)

## 5. Evaluate Model

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"F1:       {f1_score(y_test, y_pred):.3f}")
print(f"ROC AUC:  {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Not a state-of-the-art model — that's not the point here. We only need a
model that's genuinely making non-trivial predictions, so there's something
real for SHAP/LIME to explain.

## 6. Generate SHAP Explanations

In [ ]:
shap_explainer = shap.TreeExplainer(model)
shap_explanation = shap_explainer(X_test)

# For a binary RandomForestClassifier, shap_explanation.values has shape
# (n_samples, n_features, n_classes). We look at the "positive" class (1 = diabetes).
shap_values_pos = shap_explanation[:, :, 1]

shap.plots.beeswarm(shap_values_pos)

In [ ]:
# Explanation for a single patient, to compare directly against LIME below.
PATIENT_IDX = 0
print(X_test.iloc[PATIENT_IDX])
print(f"\nActual outcome: {y_test.iloc[PATIENT_IDX]}")
print(f"Predicted probability of diabetes: {y_proba[PATIENT_IDX]:.3f}")

shap.plots.waterfall(shap_values_pos[PATIENT_IDX])

## 7. Generate LIME Explanations

Explaining the same patient as above, so the two explanations are directly
comparable.

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=["no_diabetes", "diabetes"],
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

lime_exp = lime_explainer.explain_instance(
    X_test.iloc[PATIENT_IDX].values, model.predict_proba, num_features=8
)

lime_exp.as_pyplot_figure()
plt.tight_layout()
plt.show()

lime_exp.as_list()

**TODO (discussion):** compare the SHAP waterfall and LIME bar chart above
for this patient — do they rank the same features as most important, in the
same direction?

## 8. Compare Explanations Across Many Patients

A single example isn't enough to say whether SHAP and LIME generally agree.
We run both explainers over a sample of the test set and, for each patient,
compare their **top-3 most important features**.

In [ ]:
def top_k_features_shap(shap_row, feature_names, k=3):
    order = np.argsort(-np.abs(shap_row))[:k]
    return [feature_names[i] for i in order]


def top_k_features_lime(exp, k=3):
    # exp.as_list() returns (condition_str, weight) pairs, e.g. "Glucose > 140"
    ranked = sorted(exp.as_list(), key=lambda pair: -abs(pair[1]))
    top = ranked[:k]
    # crude match back to the underlying feature name
    feature_names = X_train.columns.tolist()
    matched = []
    for cond, _ in top:
        for f in feature_names:
            if f in cond:
                matched.append(f)
                break
    return matched

In [ ]:
N_SAMPLES = 50  # keep small - LIME is slow to run per-instance
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    len(X_test), size=min(N_SAMPLES, len(X_test)), replace=False
)

feature_names = X_train.columns.tolist()
agreement_rows = []

for i in sample_idx:
    shap_top3 = set(top_k_features_shap(shap_values_pos.values[i], feature_names, k=3))

    exp = lime_explainer.explain_instance(
        X_test.iloc[i].values, model.predict_proba, num_features=8
    )
    lime_top3 = set(top_k_features_lime(exp, k=3))

    overlap = shap_top3 & lime_top3
    jaccard = len(overlap) / len(shap_top3 | lime_top3) if (shap_top3 | lime_top3) else np.nan

    agreement_rows.append({
        "patient_idx": i,
        "shap_top3": sorted(shap_top3),
        "lime_top3": sorted(lime_top3),
        "overlap": sorted(overlap),
        "jaccard": jaccard,
    })

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.head(10)

In [ ]:
print(f"Mean Jaccard overlap (top-3 SHAP vs top-3 LIME): {agreement_df['jaccard'].mean():.3f}")

# How often does each feature appear in each method's top-3?
shap_freq = pd.Series(
    [f for feats in agreement_df["shap_top3"] for f in feats]
).value_counts(normalize=True)
lime_freq = pd.Series(
    [f for feats in agreement_df["lime_top3"] for f in feats]
).value_counts(normalize=True)

freq_compare = pd.DataFrame({"SHAP": shap_freq, "LIME": lime_freq}).fillna(0.0)
freq_compare.sort_values("SHAP", ascending=False)

**TODO (discussion):** which features do SHAP and LIME disagree on most?
Is the mean Jaccard overlap high or low relative to what you'd expect by
chance (with 8 features and top-3 sets, random agreement would give a
Jaccard of roughly 3/13 ≈ 0.23)?

## 9. Faithfulness: Perturbation Experiment

For a given patient, take the feature SHAP ranks as most important and
perturb it toward the population median. If the explanation is faithful to
the model, removing that feature's contribution (by moving it to a
"typical" value) should move the prediction back toward the baseline in
the direction the SHAP value implies.

In [ ]:
def perturb_top_feature_to_median(idx, X_test, shap_values_pos, X_train, model):
    row = X_test.iloc[idx].copy()
    shap_row = shap_values_pos.values[idx]
    feature_names = X_test.columns.tolist()

    top_i = int(np.argmax(np.abs(shap_row)))
    top_feature = feature_names[top_i]
    top_shap_value = shap_row[top_i]

    original_value = row[top_feature]
    median_value = X_train[top_feature].median()

    original_proba = model.predict_proba(row.to_frame().T)[0, 1]

    perturbed_row = row.copy()
    perturbed_row[top_feature] = median_value
    perturbed_proba = model.predict_proba(perturbed_row.to_frame().T)[0, 1]

    return {
        "patient_idx": idx,
        "top_feature": top_feature,
        "shap_value": top_shap_value,
        "original_value": original_value,
        "perturbed_to": median_value,
        "original_proba": original_proba,
        "perturbed_proba": perturbed_proba,
        "proba_change": perturbed_proba - original_proba,
    }


perturbation_rows = [
    perturb_top_feature_to_median(i, X_test, shap_values_pos, X_train, model)
    for i in sample_idx
]
perturbation_df = pd.DataFrame(perturbation_rows)
perturbation_df.head(10)

In [ ]:
# A SHAP value is the feature's contribution AWAY from the model's baseline
# (roughly, the average prediction). Pulling the feature to the median removes
# most of that contribution, so a faithful explanation implies proba should move
# back TOWARD the baseline - i.e. shap_value and proba_change should tend to have
# OPPOSITE signs (a positive shap_value, which pushed risk up, should see proba
# fall once that push is removed, and vice versa).
opposite_sign = np.sign(perturbation_df["shap_value"]) != np.sign(perturbation_df["proba_change"])
print(f"Fraction of patients where perturbation direction matches SHAP's implied direction: {opposite_sign.mean():.3f}")

# Plot -proba_change so that "faithful" cases fall in the top-right/bottom-left
# quadrants (same sign as shap_value), which is the intuitive reading.
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(perturbation_df["shap_value"], -perturbation_df["proba_change"], alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("SHAP value of top feature (original prediction)")
ax.set_ylabel("-1 x change in predicted probability after perturbation to median")
ax.set_title("Faithfulness check: does removing the SHAP contribution reverse it?")
plt.tight_layout()
plt.show()

**TODO (discussion):**
- Do the points fall mostly in the top-right / bottom-left quadrants of the
  plot above (faithful: removing the SHAP contribution pushes the
  prediction back the way it implied) or is there a lot of scatter into the
  other two quadrants (unfaithful)?
- Repeat this with LIME's top feature per patient (rather than SHAP's) and
  compare the two faithfulness rates.
- This is a coarse faithfulness proxy (single-feature perturbation to the
  median, which ignores feature correlations/interactions). More rigorous
  versions from the literature (see "Explainability in action" paper in the
  research doc) use fidelity/robustness/consistency metrics — worth
  implementing once we've read that paper properly.

## 10. Investigate Interesting Cases

Patients where SHAP and LIME disagreed most (lowest Jaccard overlap) are
good candidates for closer inspection.

In [ ]:
most_disagreement = agreement_df.sort_values("jaccard").head(5)
most_disagreement

**TODO:** pick one of the patients above, regenerate both explanations
(SHAP waterfall + LIME bar chart, as in sections 6-7) and look closely at
*why* they disagree — is it a genuinely ambiguous case, correlated
features being credited differently, or a limitation of one of the
methods?

## 11. Summary / Next Steps

TODO once the above has actually been run and discussed as a group:
- Headline numbers (mean Jaccard overlap, faithfulness match rate).
- Which research question(s) from `research/medical-diagnosis-xai.md` this
  best supports pursuing further.
- What would need to change to make this rigorous enough for the final
  report (e.g. proper fidelity/robustness metrics, a second dataset, a
  second model to check results aren't RF-specific).